# Design exercise D6 · Next-song recommendation — Part B

Onda's home screen shows every user a row of **ten songs they have never
played**. Your job: for each of 2,000 users, choose the ten songs they are most
likely to play for the first time in **week 9**, the week after the log ends.

You have about 15 minutes. The cells below load the data, define the metric,
validate a baseline on a held-out week and write the file. Make **your**
design choices in the cell marked *Your model*, compare them on the validation
week, then hand in.

**Scoring.** For each user, the grader counts how many of the songs they really
played for the first time in week 9 are among your ten, as a share of those
songs (recall@10), and averages over the 2,000 users. Only the grader has
week 9. 0 points at the level of ten random songs (0.003), 10 points at the
reference model's 0.149 or above, linear in between.

> **One attempt.** Only the **first** file you turn in is scored. If you
> unsubmit and turn in another file, it is not scored: you keep the score of
> the first. Choose your model with the notebook's own validation, check the file with the last
> cell, and only then turn it in.

In [ ]:
import numpy as np
import pandas as pd

BASE = "https://raw.githubusercontent.com/fabsilvestri/AIML-Course/main/design/d06/"
plays = pd.read_csv(BASE + "plays.csv")              # weeks 1 to 8
songs = pd.read_csv(BASE + "songs.csv")              # the whole catalogue
users = pd.read_csv(BASE + "users.csv")
test_users = pd.read_csv(BASE + "test_users.csv")    # the 2,000 to recommend for

assert plays["week"].between(1, 8).all() and (plays["plays"] >= 1).all()
assert set(plays["song_id"]) <= set(songs["song_id"])
assert set(test_users["user_id"]) <= set(users["user_id"])
GENRES = sorted(songs["genre"].unique())
print(f"{len(plays):,} rows: {plays['user_id'].nunique():,} users, "
      f"{plays['song_id'].nunique():,} songs played, weeks 1-8")
print(f"{len(songs):,} songs in the catalogue, "
      f"{(songs['release_week'] == 9).sum()} of them released in week 9")
plays.head()

The hand-in is scored on **first plays**: a song the user played in weeks 1–8
does not count in week 9. So the validation does the same thing one week
earlier. The model sees weeks 1–7 only, and is scored on the songs each user
played for the first time in week 8, with the metric the grader uses.

In [ ]:
def first_plays(log, week):
    """{user: songs played for the first time in `week`}, users with any."""
    before = log[log["week"] < week]
    now = log[log["week"] == week]
    seen = pd.MultiIndex.from_frame(before[["user_id", "song_id"]])
    pair = pd.MultiIndex.from_frame(now[["user_id", "song_id"]])
    return now[~pair.isin(seen)].groupby("user_id")["song_id"].apply(list).to_dict()


def recall_at_10(recs, truth):
    """Mean over users of |relevant among the ten| / |relevant|."""
    return np.mean([len(set(recs[u][:10]) & set(rel)) / len(rel)
                    for u, rel in truth.items()])


val_truth = first_plays(plays, 8)
log_7 = plays[plays["week"] <= 7]


def validate(recommend):
    """Fit on weeks 1-7, recommend for week 8, score on its first plays."""
    recs = recommend(log_7, list(val_truth), next_week=8)
    return recall_at_10(recs, val_truth)


print(f"{len(val_truth):,} users with first plays in week 8, "
      f"{np.mean([len(v) for v in val_truth.values()]):.1f} songs each on average")

A recommender here is a function `recommend(log, user_ids, next_week)`: it may
use only `log`, the catalogue and the users table, and returns ten song ids per
user. `top10` turns a score matrix (one row per user, one column per candidate
song) into the ten best per user, never a song that user has already played.

The baseline, to beat: the songs with the most listeners in the log, the same
for everybody, minus what each user already played.

In [ ]:
def top10(score, log, user_ids, cands):
    """The ten highest-scored candidates per user, played songs removed."""
    S = np.array(score, dtype=float)
    rows, cols = pd.Index(user_ids), pd.Index(cands["song_id"])
    seen = log[log["user_id"].isin(rows) & log["song_id"].isin(cols)]
    S[rows.get_indexer(seen["user_id"]), cols.get_indexer(seen["song_id"])] = -np.inf
    best = np.argsort(-S, axis=1, kind="stable")[:, :10]
    return {u: cols[best[i]].tolist() for i, u in enumerate(user_ids)}


def popular(log, user_ids, next_week):
    cands = songs[songs["release_week"] <= next_week]          # out by then
    listeners = log.groupby("song_id")["user_id"].nunique()
    pop = listeners.reindex(cands["song_id"]).fillna(0).to_numpy()
    score = np.tile(pop, (len(user_ids), 1))
    return top10(score, log, user_ids, cands)


print(f"baseline: {validate(popular):.4f}")

## Your model

Build your design here, and compare it with the baseline on the validation
week. Every choice — which weeks to count, which columns of `songs` and
`users` to use, which model — is yours, and is judged by `validate`. Keep the
signature: the hand-in below calls the same function on weeks 1–8.

In [ ]:
def recommend(log, user_ids, next_week):
    cands = songs[songs["release_week"] <= next_week]
    listeners = log.groupby("song_id")["user_id"].nunique()
    pop = listeners.reindex(cands["song_id"]).fillna(0).to_numpy()
    score = np.tile(pop, (len(user_ids), 1))          # replace with your design
    return top10(score, log, user_ids, cands)


print(f"your model: {validate(recommend):.4f}")

## Hand in

Recommend from the whole log, weeks 1–8, for week 9, and check the file before
you write it.

In [ ]:
ids = test_users["user_id"].tolist()
recs = recommend(plays, ids, next_week=9)

out = pd.DataFrame({"user_id": ids, "songs": [" ".join(recs[u]) for u in ids]})
assert len(out) == 2000 and out["user_id"].is_unique
assert out["songs"].notna().all()
lists = out["songs"].str.split()
assert (lists.map(len) == 10).all(), "exactly ten songs per user"
assert (lists.map(lambda x: len(set(x))) == 10).all(), "ten different songs"
assert set().union(*lists) <= set(songs["song_id"]), "unknown song id"
out.to_csv("recommendations.csv", index=False)
print("wrote recommendations.csv:", len(out), "rows")
out.head()

In Colab, download it (the folder icon on the left, or run the next cell),
then in Classroom open the assignment, **Add → File**, choose
`recommendations.csv`, and **Turn in**. The grader marks it within the hour.
**Only this first file counts**: a second one is not scored.

In [ ]:
try:
    from google.colab import files
    files.download("recommendations.csv")
except ImportError:
    print("Not in Colab: recommendations.csv is in the working directory.")